# NpuKit — E2E 1-layer transformer smoke (PYNQ-Z2)

Geometry: **T=8 tokens × D=8** (fits glue `MAX_LEN` and 8×8 GEMM).
Token grid is a stand-in for future **ViT/MNIST patches** (not a language model).

Fixed host scales: `SCALE_ACT=64`, `SCALE_W=64`, `SCALE_P=127`.

1. CPU/ref path (NumPy int8 GEMM + glue reference)
2. FPGA path (NPU GEMM + `npukit_glue`)
3. Compare intermediates — PASS/FAIL + tensor dumps

Keep `npukit.bit`, `.hwh`, `npukit_transformer.py`, `npukit_matmul.py` beside this notebook.
**Run All** then save so dumps stay in the file.

In [1]:
import importlib
import sys

BIT = "/home/xilinx/jupyter_notebooks/npukit.bit"
sys.path.insert(0, "/home/xilinx/jupyter_notebooks")

import npukit_transformer as nt

importlib.reload(nt)
print("E2E_T", nt.E2E_T, "E2E_D", nt.E2E_D)
print("SCALE_ACT", nt.SCALE_ACT, "SCALE_W", nt.SCALE_W, "SCALE_P", nt.SCALE_P)
print("BIT", BIT)

E2E_T 8 E2E_D 8
SCALE_ACT 64.0 SCALE_W 64.0 SCALE_P 127.0
BIT /home/xilinx/jupyter_notebooks/npukit.bit


## Offline ref path only (no bitstream)

In [2]:
rc = nt.run_e2e_smoke(bit_path=None, seed=0)
assert rc == 0
print("ref-only return", rc)

=== E2E smoke: 1-layer transformer block ===
T=8 D=8  SCALE_ACT=64.0 SCALE_W=64.0 SCALE_P=127.0
weights.wq (int8) shape=(8, 8) dtype=int8
[[  1  -1   6   1  -5   3  13   9]
 [ -7 -12  -6   0 -22  -2 -12  -7]
 [ -5  -3   4  10  -1  13  -6   3]
 [  9   1  -7  -9  -4   2 -10  -2]
 [ -2   5   2   3  -6  -1   8  14]
 [-12  15  13   8   3  -3  14  19]
 [ 17  13   3 -12   0   6 -12   4]
 [  4   7 -11  -6  -4 -11  17  -5]]

--- CPU / ref path (int8 GEMM mimicked in NumPy + glue refs) ---
x_in shape=(8, 8) dtype=int32
[[-1304  -798 -1597 -1017 -1646   602  1302  -787]
 [ -340  -952   870   146  2612 -1795   594   727]
 [ -591   956 -2357  3471 -2199  1507 -1837  1886]
 [ -630   260    87  1803  -527 -4861 -1245   301]
 [ -722  1261  1664  -242 -2441  2268  1774  -490]
 [ 3454  -572 -1863  -255  1766 -1536  3197 -1469]
 [ 1564   892  -252  1770 -2457  2224  -104  -889]
 [ 1227  1737  1261  3276  1771  2103  -885   175]]
x_in float(Q12):
[[-0.3184 -0.1948 -0.3899 -0.2483 -0.4019  0.147   0.3179 -

## Board: full block ref vs FPGA

In [3]:
rc = nt.run_e2e_smoke(bit_path=BIT, seed=0)
assert rc == 0, "E2E smoke failed"
print("board e2e return", rc)

=== E2E smoke: 1-layer transformer block ===
T=8 D=8  SCALE_ACT=64.0 SCALE_W=64.0 SCALE_P=127.0
weights.wq (int8) shape=(8, 8) dtype=int8
[[  1  -1   6   1  -5   3  13   9]
 [ -7 -12  -6   0 -22  -2 -12  -7]
 [ -5  -3   4  10  -1  13  -6   3]
 [  9   1  -7  -9  -4   2 -10  -2]
 [ -2   5   2   3  -6  -1   8  14]
 [-12  15  13   8   3  -3  14  19]
 [ 17  13   3 -12   0   6 -12   4]
 [  4   7 -11  -6  -4 -11  17  -5]]

--- CPU / ref path (int8 GEMM mimicked in NumPy + glue refs) ---
x_in shape=(8, 8) dtype=int32
[[-1304  -798 -1597 -1017 -1646   602  1302  -787]
 [ -340  -952   870   146  2612 -1795   594   727]
 [ -591   956 -2357  3471 -2199  1507 -1837  1886]
 [ -630   260    87  1803  -527 -4861 -1245   301]
 [ -722  1261  1664  -242 -2441  2268  1774  -490]
 [ 3454  -572 -1863  -255  1766 -1536  3197 -1469]
 [ 1564   892  -252  1770 -2457  2224  -104  -889]
 [ 1227  1737  1261  3276  1771  2103  -885   175]]
x_in float(Q12):
[[-0.3184 -0.1948 -0.3899 -0.2483 -0.4019  0.147   0.3179 -